In [1]:
import cv2
import mediapipe as mp
import csv
import os
import numpy as np

In [2]:
# Mediapipe Pose module
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, model_complexity=1, enable_segmentation=False)

# Main folder, under which there will be 'lunges' and 'nonlunges' folders
image_folder = "images"  # Write the path of the main folder here

# File name where we will write landmark data into the CSV file
output_file = "landmarks_output.csv"

In [3]:
def calculate_angle(a,b,c):
    radians = np.arctan2(c.y-b.y, c.x-b.x) - np.arctan2(a.y-b.y, a.x-b.x)
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

In [4]:
# Write landmark data to a file in CSV format
with open(output_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    # Iterate over 'lunges' and 'nonlunges' folders under the main folder
    for subfolder in ['rlunges', 'llunges','nonlunges','zero']:
    #for subfolder in ['tek']:
        subfolder_path = os.path.join(image_folder, subfolder)
        
        # If subfolder does not exist, skip
        if not os.path.exists(subfolder_path):
            print(f"Directory {subfolder} not found, skipping.")
            continue

        # Read images in the subfolder
        for filename in os.listdir(subfolder_path):
            if filename.endswith(".png") or filename.endswith(".jpg") or filename.endswith(".jpeg"):
                image_path = os.path.join(subfolder_path, filename)
                image = cv2.imread(image_path)

                # Convert the image to RGB (Mediapipe works in RGB format)
                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                # Perform pose estimation
                results = pose.process(image_rgb)
                # If landmarks exist, let's write them

                if results.pose_landmarks:
                    row = [filename] 

                    row.append(calculate_angle(results.pose_landmarks.landmark[12],results.pose_landmarks.landmark[24],results.pose_landmarks.landmark[26]))
                    row.append(calculate_angle(results.pose_landmarks.landmark[24],results.pose_landmarks.landmark[26],results.pose_landmarks.landmark[28]))
                    row.append(calculate_angle(results.pose_landmarks.landmark[11],results.pose_landmarks.landmark[23],results.pose_landmarks.landmark[25]))
                    row.append(calculate_angle(results.pose_landmarks.landmark[23],results.pose_landmarks.landmark[25],results.pose_landmarks.landmark[27]))
                    row.append(subfolder)

                    # Write landmark data in a single row to CSV
                    writer.writerow(row)
                else:
                    print(f"No landmarks detected in {filename}")
                    writer.writerow([filename, "No landmarks detected", subfolder])
                   
                   

print(f"Landmark data has been written to {output_file}.")

# Close the Pose module
pose.close()

Landmark data has been written to landmarks_output.csv.
